# Day 03 · 你的第一個智能助理：Agent 定義與模型設定

> 第一部・新兵入伍　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 03 - 你的第一個智能助理：Agent 定義與模型設定.md`

## 今天要學會

1. 用程式碼與 YAML 兩種方式定義 agent
2. 設定 `GenerateContentConfig`、`SafetySetting`
3. 用 `output_schema` 產生結構化輸出
4. 用 `LiteLlm` 接非 Gemini 模型

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. Agent 的三個必要成分

| 成分 | 欄位 | 給誰看 |
|---|---|---|
| **身分** | `name` / `description` | 系統 / 其他 agent |
| **指令** | `instruction` | 自己的模型 |
| **工具** | `tools` | 自己的模型 |

加上 `model`，就是一個完整的 `LlmAgent`。

In [2]:
from google.adk.agents import LlmAgent


def get_capital(country: str) -> dict:
    """查詢國家的首都。

    Args:
        country: 國家名稱，繁體中文或英文皆可。
    """
    table = {"日本": "東京", "法國": "巴黎", "澳洲": "坎培拉", "巴西": "巴西利亞"}
    return {"country": country, "capital": table.get(country, "查無資料")}


geo = LlmAgent(
    name="geo_agent",
    model=get_model(),
    description="回答各國首都與基本地理資訊。",
    instruction="你是地理助理。被問到首都時一定要呼叫 get_capital，用繁體中文簡短回答。",
    tools=[get_capital],
)

print(await run_once(geo, "澳洲的首都是哪裡？", trace=True))

  🔧 [geo_agent] 呼叫 get_capital({'country': '澳洲'})
  ↩️  [geo_agent] get_capital 回傳 {'country': '澳洲', 'capital': '坎培拉'}


  💬 [geo_agent] 澳洲的首都是坎培拉。
澳洲的首都是坎培拉。


## 2. 📌 先釐清一件文章混在一起講的事

原文把兩件事放在同一段：

> 「路線一：YAML Agent Config／路線二：程式碼定義 + 任何模型」

這容易讓人以為它們是**平行的兩種選擇**。其實不是：

| | Agent Config（YAML） | 程式碼定義 |
|---|---|---|
| 支援的模型 | **只有 Gemini** | 任何（Gemini / Claude / 本地…） |
| 狀態 | Experimental | 穩定 |
| 適合 | 快速原型、非工程師調整 | 正式開發 |

**YAML 這條路綁死 Gemini**。想接 Claude 或本地模型，就只能走程式碼。
兩者不是「風格偏好」的差別，是能力範圍的差別。

### YAML 定義實測

In [3]:
import shutil
from pathlib import Path

import yaml

WORK = Path.cwd() / "_day03"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

config = {
    "name": "yaml_agent",
    "model": "gemini-flash-lite-latest",
    "description": "用 YAML 定義的極簡 agent。",
    "instruction": "你是助理，用繁體中文回答，最多兩句話。",
}
cfg_path = WORK / "yaml_agent.yaml"
cfg_path.write_text(yaml.safe_dump(config, allow_unicode=True), encoding="utf-8")
print(cfg_path.read_text(encoding="utf-8"))

description: 用 YAML 定義的極簡 agent。
instruction: 你是助理，用繁體中文回答，最多兩句話。
model: gemini-flash-lite-latest
name: yaml_agent



In [4]:
from google.adk.agents import config_agent_utils

try:
    yaml_agent = config_agent_utils.from_config(str(cfg_path))
    print("✅ 從 YAML 載入成功:", yaml_agent.name, "|", type(yaml_agent).__name__)
    print(await run_once(yaml_agent, "用一句話說明什麼是 API"))
except Exception as exc:
    print(f"⚠️ {type(exc).__name__}: {str(exc)[:300]}")
    print("\n（Agent Config 是 Experimental 功能，API 在版本之間可能改變。")
    print("　正式專案建議走程式碼定義那條路。）")

✅ 從 YAML 載入成功: yaml_agent | LlmAgent


API 就像是軟體之間的橋樑，讓不同的應用程式能夠互相溝通並交換資料。


## 3. `generate_content_config`：控制生成行為

In [5]:
from google.genai import types


def writer(temperature: float, **kw) -> LlmAgent:
    return LlmAgent(
        name=f"w{str(temperature).replace('.', '')}",
        model=get_model(),
        instruction="為使用者給的產品寫一句廣告標語，只回標語本身。",
        generate_content_config=types.GenerateContentConfig(temperature=temperature, **kw),
    )


prompt = "一款會提醒你休息的機械鍵盤"

print("temperature=0.0（穩定，適合抽資料／走流程）")
for _ in range(2):
    print("  ", await run_once(writer(0.0), prompt))

print("\ntemperature=1.6（發散，適合創意）")
for _ in range(2):
    print("  ", await run_once(writer(1.6), prompt))

temperature=0.0（穩定，適合抽資料／走流程）


   打擊疲勞，敲出健康。


   敲醒靈感，也敲醒你的健康。

temperature=1.6（發散，適合創意）


   打擊疲勞，敲出健康。


   打擊疲勞，敲出健康。


### `max_output_tokens` 是硬切，不是風格指示

In [6]:
short = writer(0.5, max_output_tokens=40)
print(await run_once(short, "請鉅細靡遺地解釋什麼是遞迴"))

**遞迴，就是一個函式在執行的過程中，直接或間接呼叫自己的過程，直到滿足終止條件為止。**


## 4. `SafetySetting`：安全過濾強度

Gemini 內建四類安全過濾。可以逐類調整閾值。

In [7]:
safe_agent = LlmAgent(
    name="safe_agent",
    model=get_model(),
    instruction="你是助理，用繁體中文回答。",
    generate_content_config=types.GenerateContentConfig(
        safety_settings=[
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            ),
        ]
    ),
)

print("可用的 HarmCategory:")
for c in types.HarmCategory:
    if c.value.startswith("HARM_CATEGORY"):
        print("  ", c.value)
print("\n可用的閾值:")
for t in types.HarmBlockThreshold:
    print("  ", t.value)

print("\n", await run_once(safe_agent, "推薦三個適合初學者的 Python 專案"))

可用的 HarmCategory:
   HARM_CATEGORY_UNSPECIFIED
   HARM_CATEGORY_HARASSMENT
   HARM_CATEGORY_HATE_SPEECH
   HARM_CATEGORY_SEXUALLY_EXPLICIT
   HARM_CATEGORY_DANGEROUS_CONTENT
   HARM_CATEGORY_CIVIC_INTEGRITY
   HARM_CATEGORY_JAILBREAK
   HARM_CATEGORY_IMAGE_HATE
   HARM_CATEGORY_IMAGE_DANGEROUS_CONTENT
   HARM_CATEGORY_IMAGE_HARASSMENT
   HARM_CATEGORY_IMAGE_SEXUALLY_EXPLICIT

可用的閾值:
   HARM_BLOCK_THRESHOLD_UNSPECIFIED
   BLOCK_LOW_AND_ABOVE
   BLOCK_MEDIUM_AND_ABOVE
   BLOCK_ONLY_HIGH
   BLOCK_NONE
   OFF



 對於 Python 初學者來說，動手做專案是鞏固語法、理解邏輯最快的方法。以下推薦三個難易度適中、趣味性高且涵蓋不同核心概念的初學者專案：

### 1. 猜數字遊戲 (Number Guessing Game)
* **難易度**：⭐（非常適合剛學完變數、條件判斷和迴圈的初學者）
* **專案核心概念**：
  * `random` 模組的使用（隨機生成數字）
  * `while` 迴圈與 `if-elif-else` 條件判斷
  * 異常處理 (`try-except`)，防止使用者輸入非數字字元導致程式崩潰
* **專案描述**：
  電腦會隨機生成一個 1 到 100 之間的數字，玩家輸入猜測的數字。電腦會提示「太大」或「太小」，直到玩家猜中為止，最後計算玩家總共猜了幾次。
* **進階挑戰**：限制玩家的猜測次數、加入排行榜功能。

---

### 2. 簡易待辦事項清單 (To-Do List CLI)
* **難易度**：⭐⭐（適合剛學完資料結構與函式的初學者）
* **專案核心概念**：
  * 串列 (Lists) 與字典 (Dictionaries) 的應用
  * 函式 (Functions) 的定義與呼叫
  * 基本的 CRUD 操作（建立、讀取、更新、刪除）
* **專案描述**：
  這是一個命令列介面 (CLI) 程式。使用者可以選擇：
  1. 查看目前的待辦事項
  2. 新增待辦事項
  3. 刪除已完成的事項
  4. 離開程式
* **進階挑戰**：使用 Python 的 `json` 模組，讓程式關閉後能將待辦事項儲存到硬碟中，下次開啟時還能讀取。

---

### 3. 自動化檔案整理工具 (File Organizer)
* **難易度**：⭐⭐⭐（適合想了解實用工具開發、檔案管理的初學者）
* **專案核心概念**：
  * `os` 和 `shutil` 模組（操作檔案與資料夾）
  * 字串處理（取得副檔名）
  * 迴圈與路徑處理
* **專案描述**：
  我們常常把檔案堆在「下載 (Downloads)」資料夾裡導致亂七八糟。這個專案可以掃描指定的資料夾，自動根據檔案的副檔名（例如 `.jpg`, `.pdf`, `.txt`），建立對應的分類資料夾（如 Images, Documents），並

> 安全設定只是第一層。真正的護欄（把政策寫死在工具裡、用 Plugin 攔截）
> 在 **Day 28**。

## 5. 📌 補充：ADK 預設不重試

原文沒提這件事，但它是免費層一定會踩到的：**ADK 不會自動重試**。
撞到 429（配額）或 503（模型忙碌），整個執行就直接失敗。

而且**重試設定掛不上模型字串**——這是「模型字串 vs 模型物件」
最實際的差別。

In [8]:
from google.adk.models.google_llm import Gemini
from google.genai.types import HttpRetryOptions

# 寫法 A：字串。短，但沒地方放重試設定。
plain = LlmAgent(name="a", model="gemini-flash-lite-latest", instruction="hi")

# 寫法 B：物件。可以掛重試。
robust = LlmAgent(
    name="b",
    model=Gemini(
        model="gemini-flash-lite-latest",
        retry_options=HttpRetryOptions(
            attempts=5,
            initial_delay=2.0,
            max_delay=30.0,
            http_status_codes=[429, 500, 502, 503, 504],
        ),
    ),
    instruction="hi",
)

print("A（字串）:", type(plain.model).__name__, "→ 沒有 retry_options 可設")
print("B（物件）:", type(robust.model).__name__, "→", robust.model.retry_options)

A（字串）: str → 沒有 retry_options 可設
B（物件）: Gemini → attempts=5 initial_delay=2.0 max_delay=30.0 exp_base=None jitter=None http_status_codes=[429, 500, 502, 503, 504]


本教材的 `shared.get_model()` 回傳的就是寫法 B，所以每個 agent 都有重試保護。

### 模型會下架

這不是假設。`gemini-2.0-flash` 已停用：

In [9]:
try:
    dead = LlmAgent(name="dead", model="gemini-2.0-flash", instruction="hi")
    await run_once(dead, "hello")
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc)[:260]}")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'stat


所以模型 ID 要**集中管理**。本教材放在 `shared/config.py` 的 `DEFAULT_MODEL`。

## 6. `output_schema`：結構化輸出

agent 的下游是程式而不是人的時候，你要的是 JSON，不是散文。

在 instruction 裡寫「請回傳 JSON」不可靠——模型會加 ```` ```json ```` 圍籬、
會多寫一句「以下是結果」。`output_schema` 是用 Pydantic 把格式強制下來。

In [10]:
from pydantic import BaseModel, Field


class Ticket(BaseModel):
    """客服工單。"""

    category: str = Field(description="分類，只能是 退款/物流/技術/其他 之一")
    priority: int = Field(description="優先級，1 最高、5 最低")
    summary: str = Field(description="一句話摘要，繁體中文")
    needs_human: bool = Field(description="是否需要人工介入")


triager = LlmAgent(
    name="triager",
    model=get_model(),
    instruction="你是客服工單分類器。分析使用者的訊息並填寫工單。",
    output_schema=Ticket,
    output_key="ticket",
)

raw = await run_once(triager, "我的訂單三個禮拜還沒到，打電話沒人接，我要退錢！")
print("原始輸出：")
print(raw)

原始輸出：
{
  "category": "退款",
  "priority": 1,
  "summary": "使用者訂單三週未送達且客服失聯，強烈要求退款",
  "needs_human": true
}


In [11]:
ticket = Ticket.model_validate_json(raw)
print(f"分類     : {ticket.category}")
print(f"優先級   : {ticket.priority}")
print(f"摘要     : {ticket.summary}")
print(f"需要人工 : {ticket.needs_human}")

分類     : 退款
優先級   : 1
摘要     : 使用者訂單三週未送達且客服失聯，強烈要求退款
需要人工 : True


## 7. 📌 補充：`output_schema` 可以跟 `tools` 並存嗎

網路上（以及不少舊教學）會告訴你「不行，會直接拋錯」。
那在**舊版 ADK** 是對的。ADK 2.x 放寬了。實測：

In [12]:
def lookup_order(order_id: str) -> dict:
    """查詢訂單狀態。

    Args:
        order_id: 訂單編號。
    """
    return {"order_id": order_id, "status": "運送中", "eta": "2026-09-08"}


class OrderReply(BaseModel):
    order_id: str
    status: str
    message: str = Field(description="給客戶看的一句話，繁體中文")


try:
    combo = LlmAgent(
        name="combo",
        model=get_model(),
        instruction="先用 lookup_order 查詢，再依 schema 回覆。",
        tools=[lookup_order],
        output_schema=OrderReply,
    )
    print("✅ 建立成功，實跑：")
    print(await run_once(combo, "查一下訂單 A-777", trace=True))
except Exception as exc:
    print(f"❌ {type(exc).__name__}: {str(exc)[:300]}")

✅ 建立成功，實跑：


  🔧 [combo] 呼叫 lookup_order({'order_id': 'A-777'})
  ↩️  [combo] lookup_order 回傳 {'order_id': 'A-777', 'status': '運送中', 'eta': '2026-09-08'}


  🔧 [combo] 呼叫 set_model_response({'message': '您的訂單 A-777 目前狀態為「運送中」，預計於 2026-09-08 送達。', 'order_id': 'A-777', 'status': '運送中'})
  ↩️  [combo] set_model_response 回傳 {'order_id': 'A-777', 'status': '運送中', 'message': '您的訂單 A-777 目前狀態為「運送中」，預計於 2026-09-08 送達。'}
  💬 [combo] {"order_id": "A-777", "status": "運送中", "message": "您的訂單 A-777 目前狀態為「運送中」，預計於 2026-09-08 送達。"}
{"order_id": "A-777", "status": "運送中", "message": "您的訂單 A-777 目前狀態為「運送中」，預計於 2026-09-08 送達。"}


### 它是怎麼做到的

看一下上面 trace 的第二次工具呼叫——那個 **`set_model_response`** 不是你寫的。

ADK 的作法是：當 `output_schema` 和 `tools` 同時存在時，它把 schema
**包裝成一個額外的工具**塞給模型。模型先呼叫你的 `lookup_order`，
再呼叫 `set_model_response` 把結果填進 schema。

這個實作細節有兩個實際後果：

1. **會多一次工具呼叫**（也就是多一次模型往返，成本更高）。
2. **模型有可能忘記呼叫它**——它終究是一次 function call，不是硬性約束。
   純 `output_schema`（沒有 tools）走的是模型原生的 structured output，
   那個才是真的強制。

In [13]:
from google.adk.tools import set_model_response_tool

print("ADK 內建的 schema 包裝工具:", set_model_response_tool.__name__)
print()
print("目前環境 google-adk:", google.adk.__version__)
print("→ 上面的結果就是這個版本的實際行為。")
print("  跨版本／跨語言（Go、TypeScript）不保證一致，")
print("  保守做法是拆成兩個 agent：一個查、一個格式化，用 SequentialAgent 串（Day 16）。")

ADK 內建的 schema 包裝工具: google.adk.tools.set_model_response_tool

目前環境 google-adk: 2.8.0
→ 上面的結果就是這個版本的實際行為。
  跨版本／跨語言（Go、TypeScript）不保證一致，
  保守做法是拆成兩個 agent：一個查、一個格式化，用 SequentialAgent 串（Day 16）。


## 8. 接非 Gemini 模型

ADK 透過 **LiteLLM** 支援上百種模型。下面這段**不會執行**（需要你自己的端點）：

In [14]:
print('''
from google.adk.models.lite_llm import LiteLlm

# Claude（需要 ANTHROPIC_API_KEY）
LlmAgent(name="c", model=LiteLlm(model="anthropic/claude-sonnet-4-5"), instruction="...")

# 本地 Ollama —— 注意是 ollama_chat 不是 ollama
LlmAgent(name="o", model=LiteLlm(model="ollama_chat/llama3.1"), instruction="...")

# 本地 vLLM / OpenAI-compatible
LlmAgent(name="v", model=LiteLlm(
    model="openai/openai/gpt-oss-120b",   # 第一個 openai/ 是路由前綴，會被剝掉
    api_base="http://localhost:5052/v1",  # 必須含 /v1
    api_key="dummy",
), instruction="...")
''')


from google.adk.models.lite_llm import LiteLlm

# Claude（需要 ANTHROPIC_API_KEY）
LlmAgent(name="c", model=LiteLlm(model="anthropic/claude-sonnet-4-5"), instruction="...")

# 本地 Ollama —— 注意是 ollama_chat 不是 ollama
LlmAgent(name="o", model=LiteLlm(model="ollama_chat/llama3.1"), instruction="...")

# 本地 vLLM / OpenAI-compatible
LlmAgent(name="v", model=LiteLlm(
    model="openai/openai/gpt-oss-120b",   # 第一個 openai/ 是路由前綴，會被剝掉
    api_base="http://localhost:5052/v1",  # 必須含 /v1
    api_key="dummy",
), instruction="...")



### 三個實際會踩的坑

1. **Ollama 要用 `ollama_chat/` 前綴**。用 `ollama/` 會失去 function calling 能力，
   而且不會報錯——工具就是不會被呼叫。
2. **vLLM 要加 `--enable-auto-tool-choice`** 啟動參數，否則模型不發出 function call。
3. **雙前綴不是打錯字**：`openai/openai/gpt-oss-120b` 的第一個 `openai/` 是
   LiteLLM 的協定路由（會被剝掉），剩下的才是端點上真正的模型 ID。
   用 `GET /v1/models` 確認端點註冊的 ID。

### 本專案怎麼切換

`shared/config.py` 已經把兩條路都接好了。改 `.env`：

```
ADK_PROVIDER=litellm
ADK_MODEL=openai/openai/gpt-oss-120b
OPENAI_API_BASE=http://localhost:5052/v1
OPENAI_API_KEY=dummy
```

然後**所有 notebook 一行都不用改**。這就是把模型設定集中管理的價值。

In [15]:
from shared import load_settings

s = load_settings()
print(f"目前 provider : {s.provider}")
print(f"目前 model    : {s.model_name}")
print(f"get_model() 回傳型別: {type(get_model()).__name__}")

目前 provider : gemini
目前 model    : gemini-flash-lite-latest
get_model() 回傳型別: Gemini


In [16]:
shutil.rmtree(WORK, ignore_errors=True)
print("已清理")

已清理


## 9. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 跑到一半 429／503 就斷 | ADK **預設不重試**，要給 `retry_options`（只有模型物件能給） |
| 模型回 404 | 模型下架了。錯誤訊息會告訴你該換哪一個 |
| YAML 定義想接 Claude | 做不到。Agent Config **只支援 Gemini** |
| 要求回 JSON 但格式不穩 | 用 `output_schema`，不要在 instruction 裡拜託模型 |
| Ollama 工具都不被呼叫 | 前綴要用 `ollama_chat/` 不是 `ollama/` |

## 10. 動手練習

1. 幫 `Ticket` 加一個 `estimated_hours: float` 欄位，重跑第 6 節。
2. 把 `temperature` 固定成 0，連跑五次第 6 節的分類，確認結構化輸出的穩定度。
3. 拿掉 `output_schema`，改在 instruction 裡寫「請回傳 JSON」，
   跑五次，數數看有幾次能直接 `json.loads()` 成功。

## 本日回顧

- **Agent = 身分（name/description）+ 指令 + 工具 + 模型**。
- **YAML Agent Config 只支援 Gemini**，而且是 Experimental；
  要接其他模型只能走程式碼。
- **ADK 預設不重試**，而且重試設定只能掛在模型**物件**上，字串掛不了。
- **模型會下架**，模型 ID 要集中管理。
- **`output_schema` 才是可靠的結構化輸出**；在 ADK 2.8 它可以跟 `tools` 並存
  （舊版不行，網路資料多半過時），但實作方式是**把 schema 包成一個
  `set_model_response` 工具**——會多一次呼叫，而且模型有可能忘記呼叫它。
- **LiteLLM 三個坑**：`ollama_chat/` 前綴、vLLM 的 `--enable-auto-tool-choice`、雙前綴。

---
**下一天 → `../day04_runtime_web_ui/`**